# 3 — Clustering (non supervisé)

Une ligne = (année, point de grille) : 12 valeurs mensuelles de t2m et de rayonnement thermique net, plus moyenne, écart et pente.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

In [ ]:
from src import clustering as cl

clean = pd.read_parquet('../data/processed/era5_clean.parquet')
profiles = cl.build_year_profiles(clean)
profiles.head()

## Choix de k

In [ ]:
scores = cl.elbow(profiles)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
scores.plot(x='k', y='inertie', marker='o', ax=axes[0], title='Coude')
scores.plot(x='k', y='silhouette', marker='o', ax=axes[1], title='Silhouette')
plt.tight_layout();

## K-means

In [ ]:
labelled, pipe = cl.cluster(profiles)
labelled.groupby('cluster')[['t2m_mean', 't2m_amplitude', 't2m_slope', 'str_mean', 'str_amplitude']].mean().round(2)

## Profils mensuels moyens par cluster

In [ ]:
cols = [f't2m_{m:02d}' for m in range(1, 13)]
profil = labelled.groupby('cluster')[cols].mean().T
profil.index = range(1, 13)
profil.plot(figsize=(9, 4), marker='o', xlabel='mois', ylabel='t2m (°C)', title='Profil annuel moyen par cluster');

## Répartition spatiale

In [ ]:
last = labelled[labelled['year'] == labelled['year'].max()]
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=last, x='longitude', y='latitude', hue='cluster', palette='tab10', s=40, ax=ax)
ax.set_title(f'Clusters — année {int(last["year"].iloc[0])}');